# tp-20 オッズ予測モデルのモデリングとチューニング

**問い**: tp-12 の予測オッズは「固定パラメータの LightGBM 1 本（対数オッズの回帰、特徴量 = base+S+C1）」だけで、チューニングも比較もしていない。特徴量・目的変数・パラメータ・モデル族・新しい年の重み・ブレンドを比べて、どれだけ精度が上がるかを測る。**オッズ・人気を特徴量に使わない**（`assert_no_odds_features` で検査）。

## 評価（valid 2022 / 2023 / 2024。test 2025 は使わない）
- 対数誤差の RMSE: 予測した単勝オッズ（レース内で正規化して控除率を合わせたもの）と確定オッズ。
- 市場確率の誤差: レース内で 1/オッズ を正規化した市場確率 p_mkt の log の二乗誤差（レースごと）と KL(市場‖予測)。
- 順位相関（レース内 Spearman）、オッズ帯別（特に 5〜50 倍の穴馬帯）の RMSE と偏り。
- 基準（tp-12 と同じ設定）との差は、レース単位の paired bootstrap（95% 区間）。

## 比べるもの（try both）
| 群 | 変種 |
|----|------|
| 特徴量 | base+S+C1（基準）/ + 追い切り評価・厩舎コメント・血統×条件（EP） |
| 目的変数 | log(オッズ)（基準）/ log(市場確率)（レース内で正規化） |
| 損失 | 二乗 / Huber（穴馬の外れ値に頑健） |
| 新しい年の重み | なし / 半減期 1 年 / 半減期 2 年 |
| パラメータ | 基準 / ランダム探索 10 組（2023 年で選択。2023 は選択に使ったので参考） |
| モデル族 | LightGBM / CatBoost / 両者の平均 |

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow

from src.pipeline.betting import pred_odds as po, odds_model_eval as oe
from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
BET = REPO_ROOT / "data/local/modeling/betting"
OUT = BET / "pred_odds" / "tp20"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP20_THREADS", max(1, min(14, detect_compute(reserve_cores=6).threads))))
SMOKE = os.environ.get("TP20_SMOKE") == "1"
YEARS_V = [2023] if SMOKE else [2022, 2023, 2024]
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
mlflow.set_experiment("keiba-tp-odds-model")
require_tracking_server()
wp = pd.read_parquet(BET / "odds_win_place.parquet"); wp = wp[wp["win_odds"] > 0]
D = tp05_data.load_dataset(WORK)
df = D["df"]
EP, EXTRAS, PED = tp05_data.load_extras_ped(df, WORK)
feats = D["FEATS"]["base+S+C1"]
po.assert_no_odds_features(list(feats) + list(EP.columns))
T = df[["race_id", "horse_id", "date"]].reset_index().merge(wp[["race_id", "horse_id", "win_odds"]], on=["race_id", "horse_id"], how="inner")
race_all = T["race_id"].to_numpy()
T["p_mkt"] = oe.market_prob(race_all, T["win_odds"].to_numpy())
dates = pd.to_datetime(T["date"]); yr = dates.dt.year.to_numpy()
X_base = df.loc[T["index"], feats].reset_index(drop=True)
X_ep = pd.concat([X_base, EP.loc[T["index"]].reset_index(drop=True)], axis=1)
Y_odds, Y_pm = np.log(T["win_odds"].to_numpy()), np.log(T["p_mkt"].to_numpy())
print("行", len(T), "| 特徴量 base", X_base.shape[1], "| +EP", X_ep.shape[1], "| 検証年", YEARS_V, "| スレッド", THREADS)

## 学習・評価の関数

In [ ]:
P0 = {"objective": "regression", "learning_rate": 0.05, "num_leaves": 63, "min_data_in_leaf": 50, "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 1.0,
      "verbosity": -1, "num_threads": THREADS, "seed": 0, "deterministic": True, "force_row_wise": True}


def rec_w(Y, hl):
    return None if not hl else (0.5 ** (np.clip((Y - 1) - yr, 0, None) / hl))


def fit_lgb(X, target, Y, params=None, rounds=500, hl=None, es=False):
    y = Y_odds if target == "odds" else Y_pm
    tr = np.flatnonzero(dates < pd.Timestamp(f"{Y}-01-01") - pd.Timedelta(days=10))
    te = np.flatnonzero(yr == Y)
    p = dict(P0, **(params or {}))
    w = rec_w(Y, hl)
    n = rounds
    if es:
        tr_a = tr[yr[tr] < Y - 1]; va = tr[yr[tr] == Y - 1]
        b0 = lgb.train(p, lgb.Dataset(X.iloc[tr_a], label=y[tr_a], weight=None if w is None else w[tr_a]), num_boost_round=3000,
                       valid_sets=[lgb.Dataset(X.iloc[va], label=y[va])], callbacks=[lgb.early_stopping(80, verbose=False)])
        n = int(max(50, b0.best_iteration * 1.1))
    b = lgb.train(p, lgb.Dataset(X.iloc[tr], label=y[tr], weight=None if w is None else w[tr]), num_boost_round=n)
    return te, b.predict(X.iloc[te]), n


def evaluate(name, preds_by_year, target):
    # preds_by_year: {Y: (te_index, raw_prediction)}。誤差は 3 つの検証年を合算して出す
    rows, errs, kls = [], [], []
    allte, allph, allodds = [], [], []
    for Y, (te, pr) in preds_by_year.items():
        race = race_all[te]
        inv = (1.0 / np.exp(pr)) if target == "odds" else np.exp(pr)
        s = pd.Series(inv).groupby(race).transform("sum").to_numpy()
        ph = inv / s
        e = oe.per_race_sq_log_err(race, T["p_mkt"].to_numpy()[te], ph); errs.append(e)
        kls.append(oe.kl_by_race(race, T["p_mkt"].to_numpy()[te], ph))
        pred_odds = po.normalize_win_odds(race, 1.0 / ph)
        allte.append(te); allph.append(pred_odds); allodds.append(T["win_odds"].to_numpy()[te])
        rows.append({"variant": name, "year": Y, "rmse_log_odds": float(np.sqrt(((np.log(pred_odds) - np.log(T["win_odds"].to_numpy()[te])) ** 2).mean())), "mse_log_pmkt": float(e.mean()), "kl": float(kls[-1].mean())})
    return pd.DataFrame(rows), pd.concat(errs), pd.concat(kls), np.concatenate(allph), np.concatenate(allodds), np.concatenate(allte)

## 基準 → 特徴量 → 目的変数 → 損失 → 新しい年の重み

In [ ]:
RES, ERR, KLS, SERIES = [], {}, {}, {}
def run_variant(name, X, target, params=None, rounds=500, hl=None, es=False):
    t0 = time.time()
    pb, ns = {}, []
    for Y in YEARS_V:
        te, pr, n = fit_lgb(X, target, Y, params, rounds, hl, es)
        pb[Y] = (te, pr); ns.append(n)
    r, e, k, ph, od, te_all = evaluate(name, pb, target)
    r["rounds"] = int(np.mean(ns)); RES.append(r); ERR[name] = e; KLS[name] = k
    SERIES[name] = (pb, target)
    band = oe.band_table(od, ph)
    sp = oe.spearman_by_race(race_all[te_all], od, ph)
    base_e = ERR.get("V0 基準")
    d = oe.paired_bootstrap_mean_diff(e, base_e) if base_e is not None and name != "V0 基準" else {"diff": 0.0, "hi": 0.0, "lo": 0.0}
    s = r[["rmse_log_odds", "mse_log_pmkt", "kl"]].mean()
    print(f"{name}: RMSE(log odds) {s['rmse_log_odds']:.4f} | mse(log p) {s['mse_log_pmkt']:.4f} (基準差 {d['diff']:+.4f} [{d['lo']:+.4f}, {d['hi']:+.4f}]) | KL {s['kl']:.4f} | Spearman {sp:.4f} | 5〜50倍 RMSE {band.loc[[b for b in band.index if b.left in (5.0, 10.0, 20.0)], 'rmse_log'].mean():.3f} ({time.time() - t0:.0f}s)", flush=True)
    return r

run_variant("V0 基準", X_base, "odds")
run_variant("V1 +EP", X_ep, "odds")
run_variant("V2 目的=log市場確率", X_base, "pm")
run_variant("V3 +EP, log市場確率", X_ep, "pm")
run_variant("V4 +EP, log市場確率, Huber", X_ep, "pm", params={"objective": "huber", "alpha": 1.0})
for hl in (1.0, 2.0):
    run_variant(f"V5 +EP, log市場確率, 半減期{hl:g}年", X_ep, "pm", hl=hl)

## パラメータ探索（ランダム 10 組、2023 年で選択。早期終了で本数を決める）

In [ ]:
rng = np.random.default_rng(0)
cfgs = [{"num_leaves": int(rng.choice([15, 31, 63, 127, 255])), "learning_rate": float(rng.choice([0.02, 0.03, 0.05])), "min_data_in_leaf": int(rng.choice([20, 50, 100, 200])),
         "feature_fraction": float(rng.choice([0.5, 0.7, 0.9])), "lambda_l2": float(rng.choice([0.0, 1.0, 10.0, 50.0]))} for _ in range(10 if not SMOKE else 3)]
tune = []
for i, c in enumerate(cfgs):
    t0 = time.time()
    te, pr, n = fit_lgb(X_ep, "pm", 2023, c, es=True)
    race = race_all[te]
    inv = np.exp(pr); ph = inv / pd.Series(inv).groupby(race).transform("sum").to_numpy()
    mse = float(oe.per_race_sq_log_err(race, T["p_mkt"].to_numpy()[te], ph).mean())
    tune.append({**c, "rounds": n, "mse_log_pmkt_2023": mse}); print(i, c, n, round(mse, 4), f"{time.time() - t0:.0f}s", flush=True)
TUNE = pd.DataFrame(tune).sort_values("mse_log_pmkt_2023"); display(TUNE.round(4)); TUNE.to_csv(OUT / "tuning.csv", index=False)
BESTC = {k: (int(v) if k in ("num_leaves", "min_data_in_leaf") else float(v)) for k, v in TUNE.iloc[0].drop(["rounds", "mse_log_pmkt_2023"]).items()}
print("採用パラメータ:", BESTC)
run_variant("V6 +EP, log市場確率, 探索パラメータ", X_ep, "pm", params=BESTC, es=True)

## CatBoost と平均

In [ ]:
try:
    from catboost import CatBoostRegressor
    HAVE_CB = True
except Exception as ex:
    HAVE_CB = False; print("catboost なし:", ex)
if HAVE_CB:
    cat_cols = [c for c in X_ep.columns if str(X_ep[c].dtype) in ("category", "object")]
    Xc = X_ep.copy()
    for c in cat_cols:
        Xc[c] = Xc[c].astype(str)
    pb = {}
    t0 = time.time()
    for Y in YEARS_V:
        tr = np.flatnonzero(dates < pd.Timestamp(f"{Y}-01-01") - pd.Timedelta(days=10)); te = np.flatnonzero(yr == Y)
        m = CatBoostRegressor(iterations=1500 if not SMOKE else 200, learning_rate=0.06, depth=8, loss_function="RMSE", thread_count=THREADS, random_seed=0, verbose=0, cat_features=cat_cols)
        m.fit(Xc.iloc[tr], Y_pm[tr])
        pb[Y] = (te, m.predict(Xc.iloc[te]))
    r, e, k, ph, od, te_all = evaluate("V7 CatBoost +EP, log市場確率", pb, "pm")
    RES.append(r); ERR["V7 CatBoost +EP, log市場確率"] = e; KLS["V7 CatBoost +EP, log市場確率"] = k; SERIES["V7 CatBoost +EP, log市場確率"] = (pb, "pm")
    d = oe.paired_bootstrap_mean_diff(e, ERR["V0 基準"])
    print(f"V7 CatBoost: mse(log p) {r['mse_log_pmkt'].mean():.4f} (基準差 {d['diff']:+.4f} [{d['lo']:+.4f}, {d['hi']:+.4f}]) ({time.time() - t0:.0f}s)")
# 平均（log 市場確率の予測を平均）
pairs = [n for n in ("V6 +EP, log市場確率, 探索パラメータ", "V7 CatBoost +EP, log市場確率") if n in SERIES]
if len(pairs) == 2:
    pb = {Y: (SERIES[pairs[0]][0][Y][0], (SERIES[pairs[0]][0][Y][1] + SERIES[pairs[1]][0][Y][1]) / 2) for Y in YEARS_V}
    r, e, k, ph, od, te_all = evaluate("V8 LightGBM(探索)+CatBoost の平均", pb, "pm")
    RES.append(r); ERR["V8 LightGBM(探索)+CatBoost の平均"] = e; KLS["V8 LightGBM(探索)+CatBoost の平均"] = k
    d = oe.paired_bootstrap_mean_diff(e, ERR["V0 基準"])
    print(f"V8 平均: mse(log p) {r['mse_log_pmkt'].mean():.4f} (基準差 {d['diff']:+.4f} [{d['lo']:+.4f}, {d['hi']:+.4f}])")

In [ ]:
ALL = pd.concat(RES, ignore_index=True)
summ = ALL.groupby("variant", sort=False)[["rmse_log_odds", "mse_log_pmkt", "kl"]].mean()
base = ERR["V0 基準"]
summ["基準との差(mse log p)"] = [oe.paired_bootstrap_mean_diff(ERR[v], base)["diff"] for v in summ.index]
summ["95%上限"] = [oe.paired_bootstrap_mean_diff(ERR[v], base)["hi"] for v in summ.index]
summ["改善"] = summ["95%上限"] < 0
display(summ.round(4)); summ.to_csv(OUT / "odds_model_summary.csv"); ALL.to_csv(OUT / "odds_model_by_year.csv", index=False)
bestv = summ["mse_log_pmkt"].idxmin()
print("最良:", bestv)
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "smoke": SMOKE, "summary": summ.round(6).reset_index().to_dict("records"), "best": bestv, "tuned_params": BESTC, "note": "test 2025 は使っていない。2023 年はパラメータ選択に使用。"}, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
with mlflow.start_run(run_name="tp-20-odds-model", tags={"notebook": "tp-20", "git_commit": GIT_COMMIT}):
    for i_, v in enumerate(summ.index):
        mlflow.log_metrics({f"mse_log_pmkt_v{i_}": float(summ.loc[v, "mse_log_pmkt"]), f"rmse_log_odds_v{i_}": float(summ.loc[v, "rmse_log_odds"])})
    mlflow.log_dict({f"v{i_}": v for i_, v in enumerate(summ.index)}, "odds_model/metric_index.json")
    for f_ in ("odds_model_summary.csv", "odds_model_by_year.csv", "tuning.csv", "report.json"):
        if (OUT / f_).exists():
            mlflow.log_artifact(str(OUT / f_), "odds_model")